In [ ]:
import io
import copy
import os
import pandas as pd
import sys
import string

# --- Base directory (EDIT THIS) ---
# Paths below are relative to this lab-data root (the Collaboration directory).
BASE_DIR = '/home/labs/davidgo/Collaboration'   # <- set to your local copy
os.chdir(BASE_DIR)

# Read Pickle containing the PBM data

In [ ]:
path = "/home/labs/davidgo/Collaboration/USEFUL_DATASETS/Sequence/Motifs_and_TFBS/PBM/data/pbm_8mer_data.pkl"
pbm_data = pd.read_pickle(path)


## Filter TFs which we cannot use in the paper

In [ ]:
# Print the number of keys before removal
print("Number of keys before removal:", len(pbm_data))

# List of keys to remove
keys_to_remove = ['FOXO3a', 'Max']

for key in keys_to_remove:
    if key in pbm_data:
        del pbm_data[key]
    else:
        print(f"Error: Key '{key}' not found in pbm_data.")

# Print the number of keys after removal
print("Number of keys after removal:", len(pbm_data))

# Save filtered Pickle

In [ ]:
pd.to_pickle(pbm_data, "humanMPRA/TF_analysis/final/intermediate_files/filtered_PBM_pickle/pbm_8mer_data_filtered.pkl")

## Filter for genes active in osteoblasts

In [ ]:
gene_annotation_table = pd.read_csv('/home/labs/davidgo/nadavmi/usefull/Human.GRCh38.p13.annot.tsv', 
                     header=0,sep = '\t',usecols=[0,1])

In [ ]:
articular_cartilage_TPM = pd.read_csv('/home/labs/davidgo/Collaboration/USEFUL_DATASETS/Expression/Cartilage/Human/GSE114007_human_control_and_osteoarithritis_cartilage/GSE114007_norm_counts_TPM_GRCh38.p13_NCBI.tsv', 
                     header=0,sep = '\t')

articular_cartilage_TPM = articular_cartilage_TPM.iloc[:,0:9]
articular_cartilage_TPM['articular_cartilage_mean'] = articular_cartilage_TPM.iloc[:, 1:].mean(axis=1)
articular_cartilage_TPM = articular_cartilage_TPM[['GeneID','articular_cartilage_mean']]


articular_cartilage_TPM = pd.merge(articular_cartilage_TPM, gene_annotation_table, on='GeneID', how='outer') 
articular_cartilage_TPM = articular_cartilage_TPM.set_index('Symbol')
articular_cartilage_TPM = articular_cartilage_TPM[['articular_cartilage_mean']]
articular_cartilage_TPM = articular_cartilage_TPM.groupby('Symbol', as_index=True).mean() # Group by gene name and average duplicates


In [ ]:
chondrocyte_active_genes = articular_cartilage_TPM[articular_cartilage_TPM['articular_cartilage_mean'] > 1].index.tolist()


In [ ]:
len(chondrocyte_active_genes)

In [ ]:
# Print the number of keys before removal
print("Number of keys before removal:", len(pbm_data))
pbm_data_chondrocytes = pbm_data.copy()

# List of keys to remove
keys_to_keep = chondrocyte_active_genes

# Remove keys from pbm_data that do not start with any of the keys_to_keep (case-insensitive)
keys_to_delete = [
    key for key in pbm_data_chondrocytes
    if not any(key.lower().startswith(keep.lower()) for keep in keys_to_keep)
]
for key in keys_to_delete:
    del pbm_data_chondrocytes[key]

# Print the number of keys after removal
print("Number of keys after removal:", len(pbm_data_chondrocytes))




### save chondorocyte-specific PBM data

In [ ]:
pd.to_pickle(pbm_data_chondrocytes, "humanMPRA/TF_analysis/final/intermediate_files/filtered_PBM_pickle/pbm_8mer_data_filtered_chondrocytes.pkl")